> 💡 **說明**：本專案之程式碼與文檔均由 **Antigravity** 輔助編寫。

# 多元線性回歸分析報告：紅酒品質預測 (Red Wine Quality Prediction)
### **課程名稱**：人工智慧與機器學習特論 / 數據分析作業
### **學號**：4112056032 
### **姓名**：黃喻琦
### **分析架構**：CRISP-DM (Cross-Industry Standard Process for Data Mining)
### **資料集來源**：[Kaggle Red Wine Quality Dataset](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009/data)

--- 
## 📊 1. Business Understanding (商業理解)

### **商業背景與痛點**
紅酒的品質判定通常仰賴專業品酒師的感官評估（如視覺、嗅覺、味覺），這存在高度的主觀性、時間成本，且品酒師的稀缺性使得評級過程非常昂貴。對於酒莊或進口商而言，如何能在釀造或進口階段，透過**客觀的理化指標（如酸度、糖分、酒精濃度、pH值等）**來預測紅酒的品質等級，是降低品質管理成本、維持生產穩定性與制訂精準定價策略的關鍵商業課題。

### **分析目標**
本專案旨在使用**多元線性回歸 (Multiple Linear Regression, MLR)** 建立客觀的紅酒品質預測模型。我們將：
1. 探索紅酒各項理化特徵與品質之間的關聯性。
2. 使用**反向淘汰法 (Backward Elimination)** 進行統計特徵選擇，篩選出對紅酒品質具顯著影響的關鍵理化指標（顯著水準 $\alpha = 0.05$）。
3. 評估線性模型的預測能力，並繪製包含 **95% 信賴區間 (Confidence Interval, CI)** 與 **95% 預測區間 (Prediction Interval, PI)** 的精緻預測圖表，為生產者提供具統計說服力的預測範圍。

--- 
## 🔍 2. Data Understanding (資料理解)

本資料集共有 1,599 筆觀測值，包含 11 個理化特徵（自變量）與 1 個感官品質評分（因變量，分數介於 0 至 10 之間，本資料集實際範圍為 3 至 8）。

### **特徵欄位說明 (Features Description)**
1. **fixed acidity (固定酸度)**: 葡萄酒中的多數酸，與酒的結構和陳年潛力有關。
2. **volatile acidity (揮發性酸度)**: 葡萄酒中醋酸的含量，過高會產生刺鼻的醋味，影響品質。
3. **citric acid (檸檬酸)**: 少量存在，可為葡萄酒增添新鮮感和風味。
4. **residual sugar (殘糖量)**: 發酵結束後殘留的糖分，影響甜度感官。
5. **chlorides (氯化物)**: 葡萄酒中的鹽分含量，影響鹹味與口感。
6. **free sulfur dioxide (游離二氧化硫)**: 以游離狀態存在的 $\text{SO}_2$，具防腐與抗氧化作用。
7. **total sulfur dioxide (總二氧化硫)**: 游離與結合狀態二氧化硫的總和，過高會產生刺激性氣味。
8. **density (密度)**: 與酒精和糖分含量相關，水份越多密度越高，酒精越多密度越低。
9. **pH (酸鹼值)**: 描述紅酒的酸鹼度，大多數紅酒在 3.0-4.0 之間。
10. **sulphates (硫酸鹽)**: 一種葡萄酒添加劑，能轉化為二氧化硫，扮演抗菌和抗氧化劑。
11. **alcohol (酒精濃度)**: 葡萄酒的酒精百分比，顯著影響酒體的厚重感。
12. **quality (品質 - Target)**: 感官評估得分，介於 0 至 10（本資料集實際分布為 3, 4, 5, 6, 7, 8）。

下面我們將載入資料，進行敘述性統計分析並繪製特徵相關性熱圖。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 設定畫圖風格
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# 載入資料
df = pd.read_csv('winequality-red.csv')

print("===== 資料集基本維度 =====")
print(f"樣本數: {df.shape[0]}, 特徵數: {df.shape[1]}")

print("\n===== 缺漏值檢查 =====")
print(f"缺失值總數: {df.isnull().sum().sum()}")

print("\n===== 品質標籤分布 =====")
print(df['quality'].value_counts().sort_index())

# 繪製特徵相關性熱圖
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("紅酒理化特徵 - 相關性矩陣熱圖 (Correlation Matrix)", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

--- 
## 🛠️ 3. Data Preparation (資料準備)

在建模前，我們必須進行以下資料準備工作：
1. **特徵與標籤分離**：將 11 個理化特徵作為 $X$，品質分值 `quality` 作為目標變量 $y$。
2. **訓練集與測試集分割**：將資料以 **80% 訓練集 (Training Set)** 與 **20% 測試集 (Testing Set)** 進行分割。設定 `random_state=42` 以確保實驗的可重複性。
3. **資料標準化 (Standardization)**：由於各特徵單位的物理尺度差異極大（例如：密度約為 0.99，總二氧化硫則高達 289），如果不進行標準化，回歸係數的大小將無法直接比較。我們使用 `StandardScaler` 對特徵進行 $\mu=0, \sigma=1$ 的 Z-Score 標準化，這讓回歸係數（Standardized Coefficients）可以直接用來評估特徵的相對重要性。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 特徵與標籤分離
X = df.drop(columns=['quality'])
y = df['quality']

# 切分訓練與測試集 (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print(f"訓練集維度: {X_train_scaled.shape}")
print(f"測試集維度: {X_test_scaled.shape}")
print("\n標準化後的特徵平均值 (應接近0):")
print(X_train_scaled.mean().round(2)[:5])

--- 
## 🤖 4. Modeling & Feature Selection (建模與特徵選擇)

### **反向淘汰特徵選擇 (Backward Elimination)**
如果我們把所有 11 個理化特徵都放入回歸模型中，可能包含多重共線性 (Multicollinearity) 或統計上不顯著的特徵，這會降低模型的解釋力。因此，我們實作**反向淘汰演算法**：
1. 將所有特徵放進 Statsmodels OLS 模型中進行配合。
2. 找出所有特徵中 **p-value 最大且大於顯著水準 $\alpha = 0.05$** 的特徵，將其剔除。
3. 重新配合模型，重複此過程，直到所有保留下來的特徵其 p-value 皆小於或等於 0.05。

我們將逐步執行此過程並顯示篩選結果。

In [ ]:
import statsmodels.api as sm

def backward_elimination(X_data, y_data, alpha=0.05):
    features = list(X_data.columns)
    iteration = 1
    
    while len(features) > 0:
        # 加上截距項 (constant term)
        X_with_const = sm.add_constant(X_data[features])
        model = sm.OLS(y_data, X_with_const).fit()
        
        # 提取理化特徵的 p-value
        p_values = model.pvalues.drop('const')
        max_p = p_values.max()
        max_feature = p_values.idxmax()
        
        if max_p > alpha:
            print(f"  第 {iteration} 次迭代: 移除特徵 '{max_feature}' (p-value: {max_p:.4f} > {alpha})")
            features.remove(max_feature)
            iteration += 1
        else: 
            print(f"  特徵篩選完成！剩餘所有特徵的 p-value 均小於或等於 {alpha}。")
            break
    return features, model

# 執行反向淘汰
selected_features, initial_fit = backward_elimination(X_train_scaled, y_train)

print("\n最終選入的關鍵理化特徵:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

# 配合最終模型
X_train_selected = X_train_scaled[selected_features]
X_train_selected_const = sm.add_constant(X_train_selected)
final_model = sm.OLS(y_train, X_train_selected_const).fit()

print("\n" + "="*40 + " 最終 OLS 統計回歸報告 " + "="*40)
print(final_model.summary())
print("="*105)

---
## 🤖 4.5 Standardized Regression Coefficients (特徵權重與重要性分析)

為了直觀展示模型中各項理化特徵對紅酒品質預測的影響力大小及方向，我們繪製了**標準化回歸係數圖 (Coefficient Plot)**。圖中綠色代表正向影響，紅色代表負向影響，並附帶了 **95% 信心區間 (Confidence Interval) 誤差線**，這能讓我們一目了然哪些特徵最為關鍵。

In [ ]:
# 提取標準化迴歸係數與信心區間
coef_df = pd.DataFrame({
    "Feature": final_model.params.index,
    "Coefficient": final_model.params.values,
    "CI_Lower": final_model.conf_int()[0].values,
    "CI_Upper": final_model.conf_int()[1].values
})
# 移除常數截距項
coef_df = coef_df[coef_df["Feature"] != "const"].sort_values(by="Coefficient", ascending=True)

plt.figure(figsize=(10, 6))
error_left = coef_df["Coefficient"] - coef_df["CI_Lower"]
error_right = coef_df["CI_Upper"] - coef_df["Coefficient"]
errors = np.array(list(zip(error_left, error_right))).T

colors = ["#e74c3c" if x < 0 else "#2ecc71" for x in coef_df["Coefficient"]]
bars = plt.barh(coef_df["Feature"], coef_df["Coefficient"], xerr=errors, 
                color=colors, alpha=0.8, edgecolor="none", height=0.5,
                error_kw=dict(ecolor="#2c3e50", lw=1.5, capsize=4, capthick=1.5))

plt.axvline(x=0, color="#2c3e50", linestyle="-", linewidth=1, alpha=0.5)
plt.xlabel("Standardized Coefficient Weight (Impact Size)", fontsize=12, labelpad=10)
plt.ylabel("Selected 理化特徵 (Selected Features)", fontsize=12, labelpad=10)
plt.title("Standardized Regression Coefficients (Feature Weights) with 95% Confidence Bars", fontsize=14, fontweight="bold", pad=15)

for bar in bars:
    width = bar.get_width()
    plt.text(width + (0.01 if width >= 0 else -0.05), bar.get_y() + bar.get_height()/2, 
             f"{width:.3f}", 
             va="center", ha="left" if width >= 0 else "right", 
             fontsize=10, fontweight="bold", color="#2c3e50")
             
plt.tight_layout()
plt.show()

--- 
## 📈 5. Model Evaluation (模型評估)

我們使用以下經典指標在**訓練集**與**測試集**上進行模型評估，確認有無過擬合 (Overfitting) 現象：
- **決定係數 ($R^2$)**: 解釋目標變量變異數的比例。
- **調整後決定係數 (Adjusted $R^2$)**: 考量特徵數後的決定係數，懲罰無效特徵的增加。
- **平均絕對誤差 (MAE)**: 預測值與真實值之差的絕對值平均。
- **均方誤差 (MSE)**: 預測值與真實值之差的平方之平均。
- **均方根誤差 (RMSE)**: MSE 的平方根，單位與目標欄位相同，對大誤差更為敏感。

In [ ]:
from sklearn import metrics

# 準備測試集的選定特徵並加入常數項
X_test_selected = X_test_scaled[selected_features]
X_test_selected_const = sm.add_constant(X_test_selected)

# 進行預測
y_train_pred = final_model.predict(X_train_selected_const)
y_test_pred = final_model.predict(X_test_selected_const)

# 計算訓練集指標
r2_train = final_model.rsquared
adj_r2_train = final_model.rsquared_adj
mae_train = metrics.mean_absolute_error(y_train, y_train_pred)
mse_train = metrics.mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)

# 計算測試集指標
r2_test = metrics.r2_score(y_test, y_test_pred)
n_test = len(y_test)
p_features = len(selected_features)
adj_r2_test = 1 - ((1 - r2_test) * (n_test - 1) / (n_test - p_features - 1))
mae_test = metrics.mean_absolute_error(y_test, y_test_pred)
mse_test = metrics.mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)

# 彙整對比表
metrics_comparison = pd.DataFrame({
    '評估指標 (Metrics)': ['R-squared (決定係數)', 'Adjusted R-squared (調整後決定係數)', 'MAE (平均絕對誤差)', 'MSE (均方誤差)', 'RMSE (均方根誤差)'],
    '訓練集 (Train)': [r2_train, adj_r2_train, mae_train, mse_train, rmse_train],
    '測試集 (Test)': [r2_test, adj_r2_test, mae_test, mse_test, rmse_test]
}).round(4)

print("===== 訓練集與測試集評估對比 =====")
print(metrics_comparison.to_string(index=False))

--- 
## 📈 6. Visualization: Confidence & Prediction Intervals (預測圖表呈現)

本作業最核心的要求是繪製預測圖，並附帶**信賴區間 (Confidence Interval, CI)** 或 **預測區間 (Prediction Interval, PI)**。

### **統計概念辨析**
1. **信賴區間 (Confidence Interval, CI)**：代表模型的**均值預測不確定性**，即「對於一組特定的特徵值，紅酒品質*平均值*落在該區間的信心程度」。因為樣本量較大，均值的估計非常精準，因此 CI 帶通常非常窄。
2. **預測區間 (Prediction Interval, PI)**：代表模型的**單一觀測值預測不確定性**，即「對於一瓶特定的紅酒，其*個體品質*落在該區間的信心程度」。它不僅包含模型參數的估計誤差，還包含資料本身的內在隨機噪聲（Residual variance），因此 PI 帶通常比 CI 帶寬得多，在商業實踐中更能真實反映個體預測的波動程度。

下面我們將測試集樣本依預測值由小到大排序，繪製精緻的預測圖，清楚標記出真實點、預測線、95% CI 與 95% PI。

In [ ]:
# 獲取 Statsmodels 的詳細預測結果，包含 CI 與 PI
predictions_obj = final_model.get_prediction(X_test_selected_const)
predictions_summary = predictions_obj.summary_frame(alpha=0.05) # 95% 信心水準

# 整理為 DataFrame 並依預測值從小到大排序以利線條繪製
plot_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': predictions_summary['mean'].values,
    'CI_Lower': predictions_summary['mean_ci_lower'].values,
    'CI_Upper': predictions_summary['mean_ci_upper'].values,
    'PI_Lower': predictions_summary['obs_ci_lower'].values,
    'PI_Upper': predictions_summary['obs_ci_upper'].values
}).sort_values(by='Predicted').reset_index(drop=True)

# 繪圖
plt.figure(figsize=(13, 7.5))
x_indices = np.arange(len(plot_df))

# 真實值散點（略微添加垂直抖動防止點重疊）
plt.scatter(x_indices, plot_df['Actual'], color='#34495e', alpha=0.45, s=25, label='真實品質分值 (Test Actual)')

# 預測回歸線
plt.plot(x_indices, plot_df['Predicted'], color='#e74c3c', linewidth=2.5, label='多元線性回歸預測線 (MLR Fit)')

# 95% 信賴區間帶 (CI) - 預測均值的不確定性
plt.fill_between(x_indices, plot_df['CI_Lower'], plot_df['CI_Upper'], 
                 color='#f39c12', alpha=0.45, label='95% 均值信賴區間 (Confidence Interval)')

# 95% 預測區間帶 (PI) - 單一觀測值的不確定性
plt.fill_between(x_indices, plot_df['PI_Lower'], plot_df['PI_Upper'], 
                 color='#3498db', alpha=0.15, label='95% 個體預測區間 (Prediction Interval)')

plt.xlabel('測試集樣本 (依預測值排序)', fontsize=12, labelpad=10)
plt.ylabel('紅酒品質分值 (quality)', fontsize=12, labelpad=10)
plt.title('多元線性回歸預測結果 - 紅酒品質與 95% 信賴及預測區間圖', fontsize=15, fontweight='bold', pad=15)
plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='none', shadow=True, fontsize=10.5)
plt.ylim(1.5, 9.5)
plt.tight_layout()
plt.show()

---
## 📈 6.2 Model Diagnostics: Actual vs. Predicted & Residual Plot (預測精準度與殘差同質性分析)

在線性回歸模型評估中，為了確保預測結果的合理性與說服力，必須檢視兩張基礎圖表：
1. **真實值 vs. 預測值散點圖 (Actual vs. Predicted Plot)**：觀察點的集中度。我們繪製了一條代表完美預測的對角紅線 ($Y = X$) 作為比對基準。
2. **殘差 vs. 擬合值散點圖 (Residual Plot)**：這是最傳統的迴歸診斷圖，用以確認殘差在各預測水準下是否隨機且均勻分佈在 $Y = 0$ 兩側，驗證**變異數同質性 (Homoscedasticity)**。

In [ ]:
# 計算殘差
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Actual vs. Predicted Scatter
axes[0].scatter(y_test, y_test_pred, color="#2c3e50", alpha=0.5, edgecolors="none", s=40)
lims = [
    min(axes[0].get_xlim()[0], axes[0].get_ylim()[0]),
    max(axes[0].get_xlim()[1], axes[0].get_ylim()[1])
]
axes[0].plot(lims, lims, "r--", alpha=0.75, zorder=0, linewidth=2, label="Perfect Prediction (Y = X)")
axes[0].set_xlabel("Actual Quality", fontsize=11)
axes[0].set_ylabel("Predicted Quality", fontsize=11)
axes[0].set_title("Actual vs. Predicted Quality", fontsize=12, fontweight="bold")
axes[0].legend(loc="upper left")

# 2. Residuals vs. Fitted (Classic Residual Plot for Homoscedasticity)
axes[1].scatter(y_test_pred, residuals, color="#e74c3c", alpha=0.5, edgecolors="none", s=40)
axes[1].axhline(y=0, color="#2c3e50", linestyle="--", linewidth=2, alpha=0.75)
axes[1].set_xlabel("Predicted Quality (Fitted Values)", fontsize=11)
axes[1].set_ylabel("Residuals (Errors)", fontsize=11)
axes[1].set_title("Residuals vs. Fitted Values", fontsize=12, fontweight="bold")

plt.suptitle("Model Diagnostics: Prediction Accuracy & Residuals Homoscedasticity", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

---
## 📈 6.5 Residual Diagnostics (殘差分析與假設檢定)

在線性回歸分析中，我們必須驗證**殘差（誤差項）是否符合常態分佈**。若殘差不符合常態分佈，則推論統計中的 p-value 與信賴區間可能不夠精準。以下使用殘差直方圖（附常態擬合曲線）與 **Q-Q 圖 (Quantile-Quantile Plot)** 來驗證此重要假設。

In [ ]:
# 計算測試集殘差
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. 殘差直方圖與常態分布擬合曲線
sns.histplot(residuals, kde=True, ax=axes[0], color="#2c3e50", stat="density", bins=15)
from scipy.stats import norm
mu, std = norm.fit(residuals)
xmin, xmax = axes[0].get_xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, std)
axes[0].plot(x, p, "r--", linewidth=2, label=f"Normal Fit\n(μ={mu:.2f}, σ={std:.2f})")
axes[0].set_xlabel("Residuals (Errors)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("Residuals Distribution vs. Normal Curve", fontsize=12, fontweight="bold")
axes[0].legend(loc="upper right")

# 2. Q-Q 圖 (Quantile-Quantile Plot)
sm.qqplot(residuals, line="45", fit=True, ax=axes[1])
axes[1].get_lines()[0].set_markerfacecolor("#3498db")
axes[1].get_lines()[0].set_markeredgecolor("#2980b9")
axes[1].get_lines()[0].set_alpha(0.6)
axes[1].get_lines()[0].set_markersize(5)
axes[1].get_lines()[1].set_color("#e74c3c")
axes[1].get_lines()[1].set_linewidth(2)
axes[1].set_title("Normal Q-Q Plot of Residuals", fontsize=12, fontweight="bold")

plt.suptitle("Residual Diagnostics: Verifying OLS Normality Assumption", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

---
## 📖 7. NotebookLM 學術研究摘要

針對網路上關於葡萄酒品質預測的解法研究，以下為您整理的摘要與比較，這結合了從基礎線性模型到最新深度學習研究的對比：

### **葡萄酒品質預測研究摘要**

目前的葡萄酒品質預測研究，主要基於葡萄牙「Vinho Verde」數據集，利用 11 種理化指標（如酒精含量、揮發性酸、pH 值等）來預測感官品質評分。研究者普遍發現，數據集存在嚴重的**類別不平衡問題**，即普通質量的樣本遠多於極端優質或劣質的樣本。

網路上主流的解法通常將此問題視為**回歸或分類任務**。基礎方案多採用**線性回歸 (Linear Regression)** 作為基準模型，其優點在於運算快速且解釋性高，能明確指出「酒精」是影響品質的最強正向因素。為了進一步提升準確率，許多研究將 0-10 的評分轉換為二元分類（如 > 6.5 為「好酒」），並運用 **Random Forest (隨機森林)** 或 **XGBoost** 等整合學習模型，這類模型在處理非平衡數據上表現優異，被認為是目前工業實作中最具成本效益的選擇。

更進階的「更優解」則聚焦於**特徵工程與深度學習優化**。例如，採用 **SMOTE 技術** 進行上採樣以解決樣本不均問題，或利用 **1D-CNN (一維卷積神經網路)**。

1D-CNN 的優勢在於能捕捉理化指標之間的內在相關性（如 pH 值與各種酸度間的連動），這補足了傳統 DNN 忽略特徵聯繫的缺點，在實驗中展現出超越傳統機器學習模型的準確度。


---
## ⚖️ 8. 網路上主流或更優解法之比較與說明

為更直觀地呈現各技術路線的優劣，以下整理了從基礎統計模型、主流整合學習到最新深度學習研究的對比表格：

### **葡萄酒品質預測解法比較表**

| 解法類別 | 代表演算法 | 核心優勢 | 主要挑戰 / 限制 | 來源 |
| :--- | :--- | :--- | :--- | :--- |
| **基礎統計模型** | 線性回歸 (LR)、SGD | **易於解釋**特徵關係，運算成本極低；適合做為實驗基準。 | 難以處理複雜的非線性關係，預測精度較低。 | Kaggle / 學術文獻 |
| **整合學習 (主流方案)** | **隨機森林 (RF)**、Gradient Boosting、XGBoost | 預測**準確率高**，能評估特徵重要性；對不平衡數據有較好魯棒性。 | 模型較為複雜，訓練與調參（如 Optuna 搜尋）需較多運算資源。 | 數據競賽主流解法 |
| **傳統深度學習** | 多層感知器 (DNN)、PyTorch/Keras 框架 | 強大的特徵學習能力，適合處理大規模數據。 | 往往**忽略特徵間的內在聯繫**，解釋性較差。 | 學術探索方案 |
| **進階深度學習 (優解)** | **1D-CNN** | **能捕捉鄰近特徵間的相關性**（如酸度指標組），表現優於基準模型。 | 模型結構設計較為複雜，且需大量的實驗驗證。 | 最新前沿論文 |
| **數據優化技術** | SMOTE、Min-Max 正規化、PCA | 顯著改善**類別不平衡**問題，提升模型對「好酒」的辨識力。 | 處理不當可能導致過度擬合 (Overfitting)。 | 特徵工程常規技術 |

這份摘要與表格結合了從基礎線性模型到最新深度學習研究的對比，突顯了從「單純預測」轉向「理解理化特徵內在關聯」的技術趨勢。


---
## 💬 9. 與 AI 聊天紀錄

📝 本專案的完整 GPT 對話與 AI 協作開發過程，已完整導出並記錄。詳細對話軌跡與開發日誌請參閱本專案資料夾內名為**「聊天紀錄」的 PDF 檔案們**（如：聊天紀錄.pdf 或相關分割 PDF 檔）。